# 05 – Huấn luyện mô hình 

**Mục tiêu:** huấn luyện các mô hình dự đoán nhãn **Healthy / Moderate / Less Healthy** (RQ5) và chạy phân cụm K-Means (riêng, không giám sát).

**Quy trình**
1. Đọc `train.csv` và `test.csv` (đã chia 80/20 ở notebook 04) và khai báo bộ đặc trưng `FEATURES`.
2. Hai mô hình: **Logistic Regression** và **Random Forest**, mỗi mô hình nằm trong một pipeline (điền thiếu, chuẩn hóa, one-hot).
3. Cross-validation 5 lần **chỉ trên tập train** để so sánh mô hình; sau đó huấn luyện lại trên toàn bộ train và lưu mô hình.
4. Phân cụm K-Means (không giám sát).

Tập test **không được dùng ở notebook này**; việc tính các chỉ số trên test nằm ở notebook 06.


## 1. Đọc dữ liệu và thiết lập

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, f1_score, adjusted_rand_score, silhouette_score
from sklearn.cluster import KMeans

sns.set_theme(style="whitegrid")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

SEED = 42
LABELS = ["Healthy", "Moderate", "Less Healthy"]
LABEL_NUTRIENTS = ["fat_100g", "saturated-fat_100g", "sugars_100g", "salt_100g"]
# Phải trùng với FEATURES đã chọn ở notebook 04 (không chứa 4 chất tạo nhãn)
FEATURES = ["energy-kcal_100g", "proteins_100g", "fiber_100g", "additives_n", "protein_energy_share", "has_fiber"]

assert not set(LABEL_NUTRIENTS) & set(FEATURES), "Đặc trưng không được chứa 4 chất tạo nhãn"
assert not {"nutriscore_grade", "nutriscore_score", "nova_group", "fsa_points", "health_label"} & set(FEATURES)
train = pd.read_csv("train.csv", dtype={"code": str})
test = pd.read_csv("test.csv", dtype={"code": str})
y_train = train["health_label"]
print("train:", train.shape, " test:", test.shape)
print(y_train.value_counts().reindex(LABELS))

train: (6399, 20)  test: (1600, 20)
health_label
Healthy         2748
Moderate        1687
Less Healthy    1964
Name: count, dtype: int64


## 2. Pipeline, cross-validation và huấn luyện

In [2]:
def make_pipeline(features, model):
    """Điền trung vị + chuẩn hóa cho biến số, one-hot cho danh mục. Tất cả chỉ fit trên train."""
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["main_category"]),
    ])
    return Pipeline([("pre", pre), ("model", model)])

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced",
                                            random_state=SEED, n_jobs=-1),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
Xtr = train[FEATURES + ["main_category"]]

rows = []
for name, model in models.items():
    pipe = make_pipeline(FEATURES, model)
    cvr = cross_validate(pipe, Xtr, y_train, cv=cv,
                         scoring=["accuracy", "precision_macro", "recall_macro", "f1_macro"])
    rows.append({"mo_hinh": name,
                 "CV_accuracy": cvr["test_accuracy"].mean(), "CV_precision_macro": cvr["test_precision_macro"].mean(),
                 "CV_recall_macro": cvr["test_recall_macro"].mean(), "CV_f1_macro": cvr["test_f1_macro"].mean(),
                 "CV_f1_std": cvr["test_f1_macro"].std()})
    pipe.fit(Xtr, y_train)                                   # huấn luyện lại trên toàn bộ train
    joblib.dump(pipe, "model_" + name.lower().replace(" ", "_") + ".joblib")

cv_results = pd.DataFrame(rows)
print(cv_results.round(3).to_string(index=False))
cv_results.to_csv("cv_results.csv", index=False, encoding="utf-8-sig")
print("\nĐã lưu cv_results.csv và hai file mô hình .joblib")

            mo_hinh  CV_accuracy  CV_precision_macro  CV_recall_macro  CV_f1_macro  CV_f1_std
Logistic Regression        0.794               0.774            0.776        0.774      0.002
      Random Forest        0.836               0.825            0.831        0.827      0.006

Đã lưu cv_results.csv và hai file mô hình .joblib


**Nhận xét:**
- Trong cross-validation 5 lần trên tập train, **Random Forest** vượt Logistic Regression ở mọi chỉ số: accuracy 0,836 so với 0,794, F1 macro 0,827 so với 0,774. Độ lệch chuẩn F1 giữa 5 fold nhỏ (0,006 với Random Forest, 0,002 với Logistic Regression) nên kết quả ổn định.
- Random Forest tốt hơn vì quan hệ giữa năng lượng, đạm và nhãn là phi tuyến.
- Bộ đặc trưng đã loại 4 chất tạo nhãn nên các con số này không phải do mô hình học lại công thức tính nhãn. Năng lượng vẫn mang thông tin gián tiếp về béo và đường (xem notebook 07).
- Cả hai mô hình đã được huấn luyện lại trên toàn bộ train và lưu thành file `.joblib`; điểm trên tập test được tính ở notebook 06.

## 3. Phân cụm K-Means (kiểm chứng nhóm tự nhiên)

In [3]:
CL = ["energy-kcal_100g", "fat_100g", "saturated-fat_100g", "sugars_100g", "salt_100g", "proteins_100g"]
data = pd.read_csv("feature_dataset.csv", dtype={"code": str})   # K-Means không giám sát, mô tả toàn bộ 7.999 sản phẩm
Xc = StandardScaler().fit_transform(np.log1p(data[CL]))

# Chọn k bằng silhouette
ks = range(2, 7)
sil = {k: silhouette_score(Xc, KMeans(k, n_init=10, random_state=SEED).fit_predict(Xc), sample_size=4000, random_state=SEED) for k in ks}
print("Silhouette theo k:", {k: round(v, 3) for k, v in sil.items()})

km = KMeans(3, n_init=10, random_state=SEED).fit(Xc)
data["cluster"] = km.labels_
print("\nARI giữa cụm K-Means (k=3) và nhãn FSA:", round(adjusted_rand_score(data["health_label"], data["cluster"]), 3))
print("\nHồ sơ trung bình mỗi cụm:")
prof = data.groupby("cluster")[CL].mean().round(2)
prof["n"] = data["cluster"].value_counts().sort_index()
print(prof)
print("\nCụm x nhãn FSA (% theo cụm):")
print((pd.crosstab(data["cluster"], data["health_label"], normalize="index")[LABELS] * 100).round(1))
print("\nCụm x danh mục (% theo cụm):")
print((pd.crosstab(data["cluster"], data["main_category"], normalize="index") * 100).round(1))

Silhouette theo k: {2: 0.354, 3: 0.343, 4: 0.308, 5: 0.321, 6: 0.325}

ARI giữa cụm K-Means (k=3) và nhãn FSA: 0.431

Hồ sơ trung bình mỗi cụm:
         energy-kcal_100g  fat_100g  saturated-fat_100g  sugars_100g  salt_100g  proteins_100g     n
cluster                                                                                             
0                   68.31      1.50                0.61         5.98       0.25           2.93  3407
1                  297.50     16.75                7.35         2.66       1.42          11.37  2718
2                  444.81     25.60               13.08        27.32       0.35           6.65  1874

Cụm x nhãn FSA (% theo cụm):
health_label  Healthy  Moderate  Less Healthy
cluster                                      
0                87.2      12.7           0.1
1                16.0      47.2          36.8
2                 1.5      21.2          77.3

Cụm x danh mục (% theo cụm):
main_category  beverages  dairies  meals  snacks
cluster     

**Nhận xét:**
- Silhouette cao nhất ở k = 2 (0,354) và k = 3 (0,343) gần bằng nhau; chọn k = 3 để so sánh với 3 nhãn.
- 3 cụm có ý nghĩa rõ: **cụm 0** (3.407 sản phẩm) năng lượng thấp (68 kcal), phần lớn là đồ uống, sữa, meals, 87,2% Healthy; **cụm 1** (2.718) béo vừa, muối cao nhất (1,42 g) và đạm cao (11,4 g), chủ yếu meals và dairies, nhãn hỗn hợp (47,2% Moderate, 36,8% Less Healthy, 16,0% Healthy); **cụm 2** (1.874) năng lượng, béo và đường cao nhất (445 kcal, 27 g đường), 75% là snacks, 77,3% Less Healthy.
- ARI với nhãn FSA là 0,431: phân cụm không giám sát tái tạo khá tốt nhóm tự nhiên nhưng không trùng hoàn toàn với nhãn (cụm 1 trộn lẫn cả ba mức). Kết quả ủng hộ việc ba mức nhãn có cơ sở trong dữ liệu.
- Lưu ý: K-Means dùng cả 4 chất tạo nhãn (không giám sát, không dùng để dự đoán), nên ARI với nhãn FSA một phần là hiển nhiên; chỉ nên đọc như bằng chứng tham khảo rằng ba nhóm tự nhiên khớp tương đối với ba mức nhãn, không phải kiểm chứng độc lập.
- Phần so sánh đầy đủ giữa K-Means và nhãn FSA (bảng crosstab, heatmap, ARI, NMI) nằm ở mục 7 của notebook 06.